In [ ]:
from pathlib import Path
import sys

src_dir = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "common.py").is_file() and (p / "feature_preference").is_dir())
sys.path.insert(0, str(src_dir))


In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import random
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from dataset_utils import combine_and_mask
import matplotlib.pyplot as plt
import cv2
from common import *

In [ ]:
places_dir = '/home/dataset/Places365'

# Manually selected high-quality background images
land_bg_images = [
    'data_256/m/mountain/00000123.jpg',
    # 'data_256/m/mountain/00000463.jpg',
    # 'data_256/m/mountain/00000796.jpg',
    # 'data_256/m/mountain/00001011.jpg',
    # 'data_256/m/mountain/00001365.jpg'
]

water_bg_images = [
    # 'data_256/o/ocean/00000004.jpg',
    # 'data_256/o/ocean/00000008.jpg',
    # 'data_256/o/ocean/00000887.jpg',
    'data_256/o/ocean/00001120.jpg',
    # 'data_256/o/ocean/00001458.jpg'
]
# Preview background images
fig, axes = plt.subplots(2, len(water_bg_images), figsize=(20, 8))

for idx, img_path in enumerate(land_bg_images):
    img = cv2.cvtColor(cv2.imread(os.path.join(places_dir, img_path)), cv2.COLOR_BGR2RGB)
    axes[0].imshow(img)
    axes[0].set_title(f'Land {idx}')
    axes[0].axis('off')
    # axes[0, idx].imshow(img)
    # axes[0, idx].set_title(f'Land {idx}')
    # axes[0, idx].axis('off')

for idx, img_path in enumerate(water_bg_images):
    img = cv2.cvtColor(cv2.imread(os.path.join(places_dir, img_path)), cv2.COLOR_BGR2RGB)
    axes[1].imshow(img)
    axes[1].set_title(f'Water {idx}')
    axes[1].axis('off')
    # axes[1, idx].imshow(img)
    # axes[1, idx].set_title(f'Water {idx}')
    # axes[1, idx].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
bird_ratio = [0.08]
cub_dir = '/home/dataset/CUB_200_2011'
output_dir = '/home/dataset/waterbird_multiple_filtered_crop'

train_frac = 0.75  # Fraction for training set (0-1)
val_frac = 0      # Fraction for validation set (0-1)

background_predictability = 0.70
min_mask_ratio = max(bird_ratio)

In [ ]:
bias = 0.8

def get_max_bounding_box(mask):
    y_indices, x_indices = np.where(mask >= bias)
    if len(y_indices) == 0:
        return (0, 0, 0, 0)
    
    min_y, max_y = np.min(y_indices), np.max(y_indices)
    min_x, max_x = np.min(x_indices), np.max(x_indices)
    return (min_y, min_x, max_y, max_x)

def get_ratio(mask):
    bbox = get_max_bounding_box(mask)
    min_y, min_x, max_y, max_x = bbox
    height = max_y - min_y + 1
    width = max_x - min_x + 1
    area = height * width
    mask_area = np.sum(mask >= bias)
    return mask_area / area

In [ ]:
land_birds_list = ['chipping_sparrow', 'clay_colored_sparrow', 'brewer_sparrow', 'field_sparrow', 'tree_sparrow']# + ['baird_sparrow', 'fox_sparrow', 'grasshopper_sparrow']
water_birds_list = ['common_tern', 'artic_tern', 'elegant_tern', 'forsters_tern', 'caspian_tern']# + ['black_tern', 'least_tern', 'california_gull', 'glaucous_winged_gull', 'heermann_gull']
target_species = land_birds_list + water_birds_list

images_path = os.path.join(cub_dir, 'images.txt')
df = pd.read_csv(images_path, sep=" ", header=None, names=['img_id', 'img_filename'], index_col='img_id')

# Filter by target species
species_filtered_ids = []
all_species_ids = []
for img_id, row in df.iterrows():
    img_filename = row['img_filename']
    species_name = img_filename.split('/')[0].split('.')[1].lower()
    if species_name not in target_species:
        species_filtered_ids.append(img_id)
    if species_name not in all_species_ids:
        all_species_ids.append(species_name)
print(all_species_ids)

print(f"Filtering species (keeping only {len(target_species)} species): {len(species_filtered_ids)} images removed")
df = df.drop(species_filtered_ids)
print(f"After species filtering: {len(df)}")

# Filter by mask ratio
mask_ratios = []
filtered_img_ids = []

for img_id, row in tqdm(df.iterrows()):
    img_filename = row['img_filename']
    seg_path = os.path.join(cub_dir, 'segmentations', img_filename.replace('.jpg','.png'))
    try:
        seg_np = np.asarray(cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE)) / 255
        ratio = get_ratio(seg_np)
        mask_ratios.append(ratio)

        if ratio < min_mask_ratio:
            filtered_img_ids.append(img_id)
    except Exception as e:
        print(f"Error: {img_filename}")
        filtered_img_ids.append(img_id)

print(f"Filtering by mask ratio < {min_mask_ratio}: {len(filtered_img_ids)} images removed")
df = df.drop(filtered_img_ids)
print(f"After mask filtering: {len(df)}")

species = np.unique([img_filename.split('/')[0].split('.')[1].lower() for img_filename in df['img_filename']])
print(f"Final species: {species}")

In [ ]:
plt.plot(np.sort(mask_ratios))
plt.axhline(min_mask_ratio, color='r', linestyle='--')

In [ ]:
set_seed(0)

# Assign labels (water=1, land=0)
water_birds = {}
for species_name in species:
    water_birds[species_name] = 1 if species_name in water_birds_list else 0

species_list = [img_filename.split('/')[0].split('.')[1].lower() for img_filename in df['img_filename']]
df['y'] = [water_birds[species] for species in species_list]

# Custom train/val/test split (stratified by class)
df['split'] = -1
all_ids = np.array(df.index)

for y_class in [0, 1]:
    class_ids = np.array(df[df['y'] == y_class].index)
    np.random.shuffle(class_ids)
    
    n_total = len(class_ids)
    n_train = int(np.round(train_frac * n_total))
    n_val = int(np.round(val_frac * n_total))
    
    train_ids = class_ids[:n_train]
    val_ids = class_ids[n_train:n_train+n_val]
    test_ids = class_ids[n_train+n_val:]
    
    df.loc[train_ids, 'split'] = 0
    df.loc[val_ids, 'split'] = 1
    df.loc[test_ids, 'split'] = 2

train_ids = np.array(df[df['split'] == 0].index)
val_ids = np.array(df[df['split'] == 1].index)
test_ids = np.array(df[df['split'] == 2].index)

# Assign background confounders
df['place'] = 0
train_ids = np.array(df.loc[df['split'] == 0].index)
val_ids = np.array(df.loc[df['split'] == 1].index)
test_ids = np.array(df.loc[df['split'] == 2].index)

for split_idx, ids in enumerate([train_ids, val_ids, test_ids]):
    for y in (0, 1):
        pos_fraction = background_predictability if y == 1 else 1 - background_predictability
        subset_df = df.loc[ids, :]
        y_ids = np.array((subset_df.loc[subset_df['y'] == y]).index)
        pos_place_ids = np.random.choice(y_ids, size=int(np.round(pos_fraction * len(y_ids))), replace=False)
        df.loc[pos_place_ids, 'place'] = 1

for split, split_label in [(0, 'train'), (1, 'val'), (2, 'test')]:
    print(f"{split_label}:")
    split_df = df.loc[df['split'] == split, :]
    print(f"  waterbirds: {np.mean(split_df['y']):.3f}")
    print(f"  y=0,c=0: {np.mean(split_df.loc[split_df['y'] == 0, 'place'] == 0):.3f} (n={np.sum((split_df['y'] == 0) & (split_df['place'] == 0))})")
    print(f"  y=0,c=1: {np.mean(split_df.loc[split_df['y'] == 0, 'place'] == 1):.3f} (n={np.sum((split_df['y'] == 0) & (split_df['place'] == 1))})")
    print(f"  y=1,c=0: {np.mean(split_df.loc[split_df['y'] == 1, 'place'] == 0):.3f} (n={np.sum((split_df['y'] == 1) & (split_df['place'] == 0))})")
    print(f"  y=1,c=1: {np.mean(split_df.loc[split_df['y'] == 1, 'place'] == 1):.3f} (n={np.sum((split_df['y'] == 1) & (split_df['place'] == 1))})")

# Assign background images (randomly from curated list)
df['place_filename'] = ''
for idx in df.index:
    bg_list = land_bg_images if df.loc[idx, 'place'] == 0 else water_bg_images
    df.loc[idx, 'place_filename'] = random.choice(bg_list)

In [ ]:
# Random crop function for backgrounds
def random_crop(img, crop_size=144, resize_size=192):
    img = cv2.resize(img, (resize_size, resize_size))

    top = random.randint(0, resize_size - crop_size)
    left = random.randint(0, resize_size - crop_size)
    return img[top:top+crop_size, left:left+crop_size]

In [ ]:
for pref_ratio in bird_ratio:
    set_seed(0)
    dataset_name = f'{pref_ratio}'
    output_subfolder = os.path.join(output_dir, dataset_name)
    os.makedirs(output_subfolder, exist_ok=True)

    df.to_csv(os.path.join(output_subfolder, 'metadata.csv'))

    real_ratios = []
    for i in tqdm(df.index):
        # Load bird
        img_path = os.path.join(cub_dir, 'images', df.loc[i, 'img_filename'])
        seg_path = os.path.join(cub_dir, 'segmentations', df.loc[i, 'img_filename'].replace('.jpg','.png'))
        img_np = np.asarray(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB))
        seg_np = np.asarray(cv2.imread(seg_path, cv2.IMREAD_GRAYSCALE)) / 255 >= bias
        min_y, min_x, max_y, max_x = get_max_bounding_box(seg_np)
        original_ratio = get_ratio(seg_np)
        bird = img_np[min_y:max_y+1, min_x:max_x+1]
        bird_mask = seg_np[min_y:max_y+1, min_x:max_x+1]
        
        # Load and crop background
        place_path = os.path.join(places_dir, df.loc[i, 'place_filename'])
        place_full = np.asarray(cv2.cvtColor(cv2.imread(place_path), cv2.COLOR_BGR2RGB))
        place = random_crop(place_full, crop_size=144, resize_size=192)

        combined_img, real_ratio = combine_and_mask(place, bird, bird_mask, original_ratio, pref_ratio)
        real_ratios.append(real_ratio)

        output_path = os.path.join(output_subfolder, df.loc[i, 'img_filename'])
        os.makedirs('/'.join(output_path.split('/')[:-1]), exist_ok=True)
        Image.fromarray(combined_img).save(output_path)

    plt.plot(np.sort(real_ratios))
    plt.axhline(pref_ratio, color='r', linestyle='--')
    plt.title(f'Bird ratio: {pref_ratio}')
    plt.show()